# 🧬 Sturm Cellular Lifespan Study — Pipeline Reproducible
## Manifest Builder v1.0

**Proyecto:** Inferencia del envejecimiento mediante un modelo latente multimodal  
**Autor:** Juan Carlos Barajas Alarcón  
**Propósito:** Construir el manifest unificado que vincula todas las modalidades del dataset Sturm (imágenes, RNA-seq, metilación, telómero, mtDNA) a través de un `sample_id` común.  

### ⚠️ Por qué este notebook es la pieza más crítica del proyecto
El mayor riesgo no es el modelo, es el **join de IDs**. Si la imagen de un pozo no se puede vincular a su PDL, donante y medición ómica correspondiente, los encoders aprenden ruido. Este notebook resuelve eso de forma reproducible y verificable.

### Estructura del pipeline
```
PASO 1 — Setup y estructura de directorios
PASO 2 — Descarga del CSV central de biomarcadores (Figshare)
PASO 3 — Descarga de metadatos GEO (RNA-seq y Metilación)
PASO 4 — Descarga de metadatos de imágenes (Figshare)
PASO 5 — Construcción del manifest unificado
PASO 6 — Validación y reporte de calidad
PASO 7 — Export y versionado
```

---
> **NOTA para el usuario:** Si ya tienes algún archivo descargado, ve directamente al PASO 5 y apunta `RAW_PATHS` a tus archivos locales.

## PASO 1 — Setup: librerías, directorios y configuración

In [1]:
# ── Instalar dependencias ──────────────────────────────────────────────────────
!pip install GEOparse requests pandas numpy tqdm openpyxl --quiet

import os
import json
import hashlib
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas')


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/Users/JCB/Downloads/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


✅ Librerías cargadas


In [ ]:
# ── Montar Google Drive (opcional pero recomendado para persistencia) ──────────
USE_GDRIVE = True  # Cambia a False si prefieres trabajar solo en /content

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/CellAgingProject')
else:
    BASE_DIR = Path('/content/CellAgingProject')

# ── Estructura de directorios del proyecto ────────────────────────────────────
DIRS = {
    'raw':          BASE_DIR / 'data' / 'raw',
    'raw_images':   BASE_DIR / 'data' / 'raw' / 'images',
    'raw_rna':      BASE_DIR / 'data' / 'raw' / 'rna_seq',
    'raw_methyl':   BASE_DIR / 'data' / 'raw' / 'methylation',
    'raw_omics':    BASE_DIR / 'data' / 'raw' / 'omics_central',  # CSV central Figshare
    'processed':    BASE_DIR / 'data' / 'processed',
    'manifests':    BASE_DIR / 'data' / 'manifests',
    'logs':         BASE_DIR / 'logs',
    'reports':      BASE_DIR / 'reports',
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f'  📁 {name}: {path}')

print('\n✅ Estructura de directorios lista')

In [ ]:
# ── Configuración central del proyecto ────────────────────────────────────────
# Fuentes de datos conocidas para el dataset Sturm et al. 2022
CONFIG = {
    # Figshare: dataset central de biomarcadores (telómero, mtDNA, bioenergética, citología, metadatos)
    'figshare_central_article_id': '18441998',
    'figshare_central_version': '2',

    # Figshare: imágenes brightfield
    'figshare_images_article_id': '18444731',
    'figshare_images_version': '1',

    # GEO accessions
    'geo_rna':        'GSE179848',  # RNA-seq bulk
    'geo_methylation': 'GSE179847', # Metilación EPIC array
    'geo_superseries': 'GSE179849', # Superseries (referencia general)

    # GitHub (código oficial del estudio — fuente de verdad para IDs)
    'github_repo': 'https://raw.githubusercontent.com/gav-sturm/Cellular_Lifespan_Study/main',

    # Shiny app (para referencia)
    'shiny_app': 'https://columbia-picard.shinyapps.io/shinyapp-Lifespan_Study/',

    # Versión del pipeline
    'pipeline_version': '1.0.0',
    'pipeline_date': datetime.now().strftime('%Y-%m-%d'),
}

# Guardar config
with open(DIRS['logs'] / 'pipeline_config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)

print('✅ Configuración guardada')
print(json.dumps(CONFIG, indent=2))

## PASO 2 — Descarga del CSV central de biomarcadores (Figshare)

Este archivo es la **ancla de todo el proyecto**: contiene `sample_id`, `donor_id`, `passage`, `PDL`, `treatment`, `phase`, telómero, mtDNA copy number, y biomarcadores por muestra.

Es la fuente de verdad para el join con el resto de modalidades.

In [ ]:
# ── Utilidades de descarga con checksum ───────────────────────────────────────
def compute_md5(filepath):
    """Calcula el MD5 de un archivo para verificación."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def download_file(url, dest_path, description=''):
    """Descarga con barra de progreso. Retorna (path, md5)."""
    dest = Path(dest_path)
    if dest.exists():
        md5 = compute_md5(dest)
        print(f'  ⏭️  Ya existe: {dest.name} (md5: {md5[:8]}...)')
        return dest, md5

    print(f'  ⬇️  Descargando: {description or url}')
    resp = requests.get(url, stream=True)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))

    with open(dest, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as bar:
        for chunk in resp.iter_content(8192):
            f.write(chunk)
            bar.update(len(chunk))

    md5 = compute_md5(dest)
    print(f'  ✅ Guardado: {dest.name} | md5: {md5[:8]}... | size: {dest.stat().st_size/1e6:.1f} MB')
    return dest, md5

def list_figshare_files(article_id, version=None):
    """Lista todos los archivos de un artículo de Figshare."""
    if version:
        url = f'https://api.figshare.com/v2/articles/{article_id}/versions/{version}/files'
    else:
        url = f'https://api.figshare.com/v2/articles/{article_id}/files'
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json()

print('✅ Utilidades de descarga definidas')

In [ ]:
# ── Listar archivos disponibles en Figshare (dataset central) ─────────────────
print('📋 Archivos disponibles en Figshare (dataset central Sturm):')
print('=' * 70)

central_files = list_figshare_files(
    CONFIG['figshare_central_article_id'],
    CONFIG['figshare_central_version']
)

figshare_catalog = {}
for f in central_files:
    size_mb = f.get('size', 0) / 1e6
    figshare_catalog[f['name']] = {
        'id': f['id'],
        'url': f['download_url'],
        'size_mb': round(size_mb, 2),
        'md5': f.get('computed_md5', 'N/A')
    }
    print(f"  [{size_mb:6.1f} MB]  {f['name']}")

# Guardar catálogo
with open(DIRS['logs'] / 'figshare_central_catalog.json', 'w') as f:
    json.dump(figshare_catalog, f, indent=2)

print(f'\n✅ {len(central_files)} archivos encontrados. Catálogo guardado.')

In [ ]:
# ── Descargar archivos clave del dataset central ───────────────────────────────
# Archivos prioritarios para el manifest (ajusta los nombres si el catálogo muestra nombres distintos)
CENTRAL_FILES_TO_DOWNLOAD = [
    # Estos son los nombres típicos del repositorio Sturm — verificar contra el catálogo arriba
    'Lifespan_Data.csv',          # Tabla principal: sample_id, donor, PDL, biomarcadores
    'Lifespan_Metadata.csv',      # Metadatos de donantes y experimentos
    'telomere_data.csv',          # Longitud telomérica por muestra
    'mtDNA_data.csv',             # mtDNA copy number por muestra
    'cytology_data.csv',          # Datos morfológicos/citología
]

downloaded_central = {}

for target_name in CENTRAL_FILES_TO_DOWNLOAD:
    # Buscar el archivo en el catálogo (coincidencia parcial de nombre)
    matches = [(name, info) for name, info in figshare_catalog.items()
               if target_name.lower() in name.lower() or
                  any(kw in name.lower() for kw in target_name.lower().split('_'))]

    if not matches:
        print(f'  ⚠️  No encontrado: {target_name} — revisar catálogo manualmente')
        continue

    # Tomar el primer match
    matched_name, info = matches[0]
    dest = DIRS['raw_omics'] / matched_name
    path, md5 = download_file(info['url'], dest, matched_name)
    downloaded_central[matched_name] = {'path': str(path), 'md5': md5}

print(f'\n✅ {len(downloaded_central)} archivos centrales descargados')

# También descargar TODOS los archivos del catálogo que sean CSV/xlsx (seguros de tamaño)
print('\n📥 Descargando todos los archivos tabulares del dataset central...')
for name, info in figshare_catalog.items():
    if info['size_mb'] < 100 and any(name.endswith(ext) for ext in ['.csv', '.xlsx', '.tsv', '.txt']):
        dest = DIRS['raw_omics'] / name
        path, md5 = download_file(info['url'], dest, name)
        downloaded_central[name] = {'path': str(path), 'md5': md5}

# Guardar registro de descargas
with open(DIRS['logs'] / 'downloaded_central.json', 'w') as f:
    json.dump(downloaded_central, f, indent=2)

## PASO 3 — Descarga de metadatos GEO (RNA-seq y Metilación)

GEOparse descarga los metadatos de las muestras (GSM), que son los que nos dan los `sample_id` para el join. **No** descargamos los FASTQ/BAM crudos aquí (eso se hace en el pipeline de procesamiento por modalidad). Solo los matrices de expresión procesadas y los metadatos.

In [ ]:
import GEOparse

def fetch_geo_metadata(accession, dest_dir):
    """
    Descarga metadatos de GEO y extrae tabla de muestras.
    Retorna un DataFrame con sample_id, title, características y suplementarios.
    """
    print(f'\n📡 Descargando metadatos GEO: {accession}')
    gse = GEOparse.get_GEO(
        geo=accession,
        destdir=str(dest_dir),
        silent=True
    )

    rows = []
    for gsm_name, gsm in gse.gsms.items():
        row = {
            'geo_sample_id': gsm_name,
            'title': gsm.metadata.get('title', [''])[0],
            'source_name': gsm.metadata.get('source_name_ch1', [''])[0],
            'organism': gsm.metadata.get('organism_ch1', [''])[0],
            'geo_accession': accession,
        }
        # Extraer características (donor, PDL, treatment, etc. suelen estar aquí)
        for char in gsm.metadata.get('characteristics_ch1', []):
            if ':' in char:
                k, v = char.split(':', 1)
                row[f'char_{k.strip().lower().replace(" ", "_")}'] = v.strip()

        # URLs de archivos suplementarios
        supp = gsm.metadata.get('supplementary_file', [])
        row['supplementary_files'] = '|'.join(supp)
        rows.append(row)

    df = pd.DataFrame(rows)
    print(f'  ✅ {len(df)} muestras encontradas')
    print(f'  📋 Columnas: {list(df.columns)}')
    return df, gse

print('✅ Función GEO definida')

In [ ]:
# ── Descargar metadatos RNA-seq (GSE179848) ───────────────────────────────────
df_rna_meta, gse_rna = fetch_geo_metadata(CONFIG['geo_rna'], DIRS['raw_rna'])

# Guardar
rna_meta_path = DIRS['raw_rna'] / 'GSE179848_sample_metadata.csv'
df_rna_meta.to_csv(rna_meta_path, index=False)
print(f'\n💾 Metadatos RNA-seq guardados: {rna_meta_path}')
df_rna_meta.head(5)

In [ ]:
# ── Descargar metadatos Metilación (GSE179847) ────────────────────────────────
df_methyl_meta, gse_methyl = fetch_geo_metadata(CONFIG['geo_methylation'], DIRS['raw_methyl'])

# Guardar
methyl_meta_path = DIRS['raw_methyl'] / 'GSE179847_sample_metadata.csv'
df_methyl_meta.to_csv(methyl_meta_path, index=False)
print(f'\n💾 Metadatos Metilación guardados: {methyl_meta_path}')
df_methyl_meta.head(5)

In [ ]:
# ── Inspeccionar columnas de características para identificar campos de PDL/donante ──
print('=== Características en RNA-seq samples ===')
char_cols_rna = [c for c in df_rna_meta.columns if c.startswith('char_')]
print(f'Campos: {char_cols_rna}')
if char_cols_rna:
    display(df_rna_meta[char_cols_rna].head(10))

print('\n=== Características en Metilación samples ===')
char_cols_methyl = [c for c in df_methyl_meta.columns if c.startswith('char_')]
print(f'Campos: {char_cols_methyl}')
if char_cols_methyl:
    display(df_methyl_meta[char_cols_methyl].head(10))

## PASO 4 — Catálogo de imágenes (Figshare)

Las imágenes son ~4,000 archivos. **No** las descargamos todas aquí — eso requiere espacio. Lo que sí construimos es un **catálogo de nombres de archivo** que nos permita hacer el join con el resto de modalidades sin necesidad de tenerlas localmente.

In [ ]:
# ── Listar archivos de imágenes en Figshare ───────────────────────────────────
print('📸 Listando archivos de imágenes en Figshare...')
image_files = list_figshare_files(
    CONFIG['figshare_images_article_id'],
    CONFIG['figshare_images_version']
)

# Parsear nombres de archivo para extraer metadatos codificados
# Naming convention típica en estos datasets: DonorID_Passage_Magnification_field.tif
image_records = []
for f in image_files:
    name = f['name']
    record = {
        'filename': name,
        'figshare_file_id': f['id'],
        'download_url': f['download_url'],
        'size_mb': round(f.get('size', 0) / 1e6, 3),
        'md5_figshare': f.get('computed_md5', 'N/A'),
    }

    # Intentar parsear el nombre para extraer donor/passage/magnification
    # AJUSTAR este bloque una vez que veamos los nombres reales
    stem = Path(name).stem
    parts = stem.split('_')

    # Heurística inicial — revisar y ajustar con los nombres reales
    record['name_parts'] = parts
    record['n_parts'] = len(parts)

    # Detectar magnification (10x, 20x)
    for p in parts:
        if p.lower() in ['10x', '20x', '10', '20']:
            record['magnification'] = p
            break
    else:
        record['magnification'] = 'unknown'

    image_records.append(record)

df_images = pd.DataFrame(image_records)

# Guardar catálogo de imágenes
img_catalog_path = DIRS['raw_images'] / 'figshare_image_catalog.csv'
df_images.to_csv(img_catalog_path, index=False)

print(f'\n✅ {len(df_images)} imágenes catalogadas')
print(f'   Tamaño total estimado: {df_images["size_mb"].sum():.0f} MB')
print(f'   Catálogo guardado: {img_catalog_path}')
print(f'\n📋 Primeros nombres de archivo:')
for name in df_images['filename'].head(15).tolist():
    print(f'   {name}')

In [ ]:
# ── Análisis del naming convention de las imágenes ────────────────────────────
# ESTE BLOQUE ES CRÍTICO: entender la estructura del nombre de archivo
# para poder extraer donor_id, passage, magnification

print('🔍 Análisis de estructura de nombres de archivo:')
print('=' * 60)

# Mostrar distribución de número de partes
n_parts_dist = df_images['n_parts'].value_counts().sort_index()
print('\nDistribución de partes en nombre de archivo:')
print(n_parts_dist.to_string())

# Mostrar ejemplos por grupo
for n in n_parts_dist.index[:3]:
    examples = df_images[df_images['n_parts'] == n]['filename'].head(3).tolist()
    print(f'\n  {n} partes:')
    for ex in examples:
        print(f'    {ex}')

# Distribución de magnification detectada
print('\nDistribución de magnificación:')
print(df_images['magnification'].value_counts().to_string())

## PASO 5 — Construcción del Manifest Unificado

El manifest es el **contrato de datos** del proyecto. Cada fila = una muestra (`sample_id`). Las columnas de `has_*` indican qué modalidades están disponibles para esa muestra — esto es lo que habilita el missingness explícito en los encoders multimodales.

In [ ]:
# ── Cargar el CSV central de Figshare (tabla principal) ───────────────────────
# Buscar el archivo central en raw_omics
csv_files = list(DIRS['raw_omics'].glob('*.csv'))
print('📂 Archivos CSV disponibles en raw_omics:')
for f in csv_files:
    print(f'   {f.name} ({f.stat().st_size/1e3:.1f} KB)')

# Cargar el archivo que parezca ser la tabla maestra
# AJUSTAR este nombre una vez que se descargue y se vea el catálogo real
MAIN_CSV_KEYWORDS = ['lifespan', 'data', 'master', 'metadata', 'sample']

df_central = None
for f in csv_files:
    if any(kw in f.name.lower() for kw in MAIN_CSV_KEYWORDS):
        try:
            df_central = pd.read_csv(f)
            print(f'\n✅ Cargado como tabla central: {f.name}')
            print(f'   Shape: {df_central.shape}')
            print(f'   Columnas: {list(df_central.columns)}')
            break
        except Exception as e:
            print(f'   ⚠️  Error cargando {f.name}: {e}')

if df_central is None:
    print('⚠️  No se encontró tabla central automáticamente.')
    print('    Revisar catálogo y cargar manualmente:')
    print('    df_central = pd.read_csv(DIRS["raw_omics"] / "NOMBRE_DEL_ARCHIVO.csv")')

In [ ]:
# ── Normalización de columnas clave ────────────────────────────────────────────
# Este bloque mapea los nombres reales de columnas del dataset Sturm
# a los nombres estandarizados del proyecto.

# MAPEO A AJUSTAR: columna_real_en_sturm -> nombre_estandarizado
# Basado en el paper y el repositorio GitHub de Sturm
COLUMN_MAP = {
    # IDs
    'SampleID':     'sample_id',
    'Sample_ID':    'sample_id',
    'sample_id':    'sample_id',
    'DonorID':      'donor_id',
    'Donor_ID':     'donor_id',
    'donor_id':     'donor_id',
    'SubjectID':    'donor_id',

    # Tiempo y edad replicativa
    'PDL':           'pdl',
    'cumPDL':        'pdl',
    'cum_PDL':       'pdl',
    'Passage':       'passage',
    'passage':       'passage',
    'Timepoint':     'timepoint',
    'Phase':         'phase',
    'phase':         'phase',

    # Condición experimental
    'Treatment':     'treatment',
    'treatment':     'treatment',
    'Condition':     'condition',
    'condition':     'condition',
    'Group':         'group',

    # Biomarcadores (anclas de hallmarks)
    'telomere_length':      'telomere_length',
    'TelomereLength':       'telomere_length',
    'mtDNA_CN':             'mtdna_cn',
    'mtDNA_copy_number':    'mtdna_cn',

    # Citología
    'cell_size':            'cell_size',
    'CellSize':             'cell_size',
}

def normalize_columns(df, col_map):
    """Renombra columnas según el mapa, ignorando las que no están."""
    rename_dict = {k: v for k, v in col_map.items() if k in df.columns}
    return df.rename(columns=rename_dict)

print('✅ Mapeo de columnas definido')
print(f'   {len(COLUMN_MAP)} mapeos configurados')

In [ ]:
# ── Construcción del Manifest ─────────────────────────────────────────────────
def build_manifest(df_central, df_rna_meta, df_methyl_meta, df_images, col_map):
    """
    Construye el manifest unificado. Cada fila = una muestra.
    Las columnas has_* indican disponibilidad por modalidad.

    Returns:
        DataFrame con el manifest completo y flags de disponibilidad.
    """
    print('🔨 Construyendo manifest unificado...')

    # 1. Base: tabla central normalizada
    if df_central is not None:
        df_base = normalize_columns(df_central.copy(), col_map)
    else:
        # Si no hay tabla central, construir desde GEO
        print('  ⚠️  Sin tabla central — usando metadatos GEO como base')
        df_base = normalize_columns(df_rna_meta.copy(), col_map)
        df_base.rename(columns={'geo_sample_id': 'sample_id'}, inplace=True)

    # Asegurar que sample_id existe
    if 'sample_id' not in df_base.columns:
        print('  ⚠️  No se encontró sample_id. Columnas disponibles:')
        print(f'      {list(df_base.columns)}')
        print('  → Asignando índice como sample_id temporal')
        df_base['sample_id'] = df_base.index.astype(str)

    df_base = df_base.drop_duplicates(subset=['sample_id'])
    print(f'  📊 Base: {len(df_base)} muestras únicas')

    # 2. Columnas mínimas requeridas con valores por defecto
    for col in ['donor_id', 'pdl', 'passage', 'phase', 'treatment', 'condition']:
        if col not in df_base.columns:
            df_base[col] = np.nan
            print(f'  ⚠️  Columna faltante: {col} → añadida como NaN')

    # 3. Flag de disponibilidad RNA-seq
    if df_rna_meta is not None and len(df_rna_meta) > 0:
        # Intentar join por campos de características
        rna_ids = set(df_rna_meta.get('geo_sample_id', pd.Series()).tolist())
        # Buscar campo de sample_id en características GEO
        rna_char_ids = set()
        for col in df_rna_meta.columns:
            if 'sample' in col.lower() or 'id' in col.lower():
                rna_char_ids.update(df_rna_meta[col].dropna().astype(str).tolist())

        df_base['geo_rna_sample_id'] = df_base['sample_id'].apply(
            lambda x: x if str(x) in rna_char_ids else None
        )
        df_base['has_rna'] = df_base['geo_rna_sample_id'].notna()
        print(f'  🧬 RNA-seq: {df_base["has_rna"].sum()} muestras con datos')
    else:
        df_base['has_rna'] = False
        df_base['geo_rna_sample_id'] = None

    # 4. Flag de disponibilidad Metilación
    if df_methyl_meta is not None and len(df_methyl_meta) > 0:
        methyl_char_ids = set()
        for col in df_methyl_meta.columns:
            if 'sample' in col.lower() or 'id' in col.lower():
                methyl_char_ids.update(df_methyl_meta[col].dropna().astype(str).tolist())

        df_base['geo_methyl_sample_id'] = df_base['sample_id'].apply(
            lambda x: x if str(x) in methyl_char_ids else None
        )
        df_base['has_methylation'] = df_base['geo_methyl_sample_id'].notna()
        print(f'  🔬 Metilación: {df_base["has_methylation"].sum()} muestras con datos')
    else:
        df_base['has_methylation'] = False
        df_base['geo_methyl_sample_id'] = None

    # 5. Flag de disponibilidad Telómero y mtDNA (del CSV central)
    df_base['has_telomere'] = df_base.get('telomere_length', pd.Series(np.nan, index=df_base.index)).notna()
    df_base['has_mtdna']    = df_base.get('mtdna_cn', pd.Series(np.nan, index=df_base.index)).notna()
    print(f'  📍 Telómero: {df_base["has_telomere"].sum()} muestras con datos')
    print(f'  🔋 mtDNA: {df_base["has_mtdna"].sum()} muestras con datos')

    # 6. Flag de disponibilidad Imágenes
    if df_images is not None and len(df_images) > 0:
        # Extraer sample_ids potenciales de nombres de archivo
        # AJUSTAR la lógica según la naming convention real
        image_sample_ids = set()
        for filename in df_images['filename']:
            stem = Path(filename).stem
            image_sample_ids.add(stem)
            # También agregar partes separadas por _ como candidatos a IDs
            image_sample_ids.update(stem.split('_'))

        df_base['has_image'] = df_base['sample_id'].apply(
            lambda x: str(x) in image_sample_ids
        )
        df_base['n_images'] = df_base['sample_id'].apply(
            lambda x: df_images['filename'].str.contains(str(x), regex=False).sum()
            if len(str(x)) > 3 else 0
        )
        print(f'  📸 Imágenes: {df_base["has_image"].sum()} muestras con imágenes')
    else:
        df_base['has_image'] = False
        df_base['n_images'] = 0

    # 7. Columna de modalidades disponibles (para debug y filtrado rápido)
    modality_cols = ['has_image', 'has_rna', 'has_methylation', 'has_telomere', 'has_mtdna']
    df_base['n_modalities'] = df_base[modality_cols].sum(axis=1)
    df_base['modalities_available'] = df_base[modality_cols].apply(
        lambda row: '|'.join([c.replace('has_', '') for c in modality_cols if row[c]]),
        axis=1
    )

    # 8. PDL normalizado (0-1 por donante) — útil para el encoder
    if 'pdl' in df_base.columns and df_base['pdl'].notna().sum() > 0 and 'donor_id' in df_base.columns:
        df_base['pdl_norm'] = df_base.groupby('donor_id')['pdl'].transform(
            lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
        )
        df_base['pdl_bin'] = pd.cut(
            df_base['pdl_norm'],
            bins=[0, 0.33, 0.67, 1.01],
            labels=['early', 'mid', 'late'],
            include_lowest=True
        )
        print(f'  📊 PDL normalizado creado')
    else:
        df_base['pdl_norm'] = np.nan
        df_base['pdl_bin'] = np.nan

    # 9. Metadatos del pipeline
    df_base['manifest_version'] = CONFIG['pipeline_version']
    df_base['manifest_date'] = CONFIG['pipeline_date']

    return df_base

# Ejecutar construcción del manifest
df_manifest = build_manifest(
    df_central=df_central,
    df_rna_meta=df_rna_meta,
    df_methyl_meta=df_methyl_meta,
    df_images=df_images,
    col_map=COLUMN_MAP
)

print(f'\n✅ Manifest construido: {df_manifest.shape}')

## PASO 6 — Validación y Reporte de Calidad

In [ ]:
# ── Validación del manifest ────────────────────────────────────────────────────
def validate_manifest(df):
    """
    Valida el manifest y genera un reporte de calidad.
    Las validaciones están basadas en los riesgos identificados en el análisis
    de factibilidad del MVP-1.
    """
    issues = []
    warnings_list = []
    stats = {}

    print('🔍 Validando manifest...')
    print('=' * 60)

    # ── CHECK 1: IDs únicos ──
    n_dup = df['sample_id'].duplicated().sum()
    if n_dup > 0:
        issues.append(f'❌ {n_dup} sample_ids duplicados — revisar tabla central')
    else:
        print(f'  ✅ sample_id: {len(df)} únicos, sin duplicados')
    stats['n_samples'] = len(df)
    stats['n_duplicate_ids'] = int(n_dup)

    # ── CHECK 2: PDL disponible ──
    pdl_missing = df['pdl'].isna().sum()
    pdl_pct = pdl_missing / len(df) * 100
    if pdl_missing > 0:
        warnings_list.append(f'⚠️  PDL faltante en {pdl_missing} muestras ({pdl_pct:.1f}%)')
    else:
        print(f'  ✅ PDL: disponible en 100% de muestras')
    stats['pdl_missing'] = int(pdl_missing)

    # ── CHECK 3: donor_id disponible ──
    donor_missing = df['donor_id'].isna().sum()
    if donor_missing > 0:
        issues.append(f'❌ donor_id faltante en {donor_missing} muestras — alto riesgo de leakage')
    else:
        n_donors = df['donor_id'].nunique()
        print(f'  ✅ donor_id: {n_donors} donantes únicos, sin faltantes')
    stats['n_donors'] = int(df['donor_id'].nunique()) if 'donor_id' in df else 0

    # ── CHECK 4: Distribución de modalidades ──
    print(f'\n  📊 Disponibilidad de modalidades:')
    modality_cols = ['has_image', 'has_rna', 'has_methylation', 'has_telomere', 'has_mtdna']
    for col in modality_cols:
        if col in df.columns:
            n = df[col].sum()
            pct = n / len(df) * 100
            bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
            print(f'     {col.replace("has_", ""):12s}: {bar} {n:4d} ({pct:.0f}%)')
            stats[col] = int(n)

    # ── CHECK 5: Muestras con múltiples modalidades (para entrenamiento multimodal) ──
    print(f'\n  🔗 Muestras por número de modalidades disponibles:')
    if 'n_modalities' in df.columns:
        mod_dist = df['n_modalities'].value_counts().sort_index()
        for n_mod, count in mod_dist.items():
            bar = '█' * count if count < 50 else '█' * 50 + f'...({count})'
            print(f'     {n_mod} modalidades: {count:4d} muestras')
        stats['multimodal_samples'] = int((df['n_modalities'] >= 2).sum())

    # ── CHECK 6: PDL range y posibles outliers ──
    if df['pdl'].notna().sum() > 0:
        pdl_stats = df['pdl'].describe()
        print(f'\n  📈 Estadísticas de PDL:')
        print(f'     Min: {pdl_stats["min"]:.1f} | Max: {pdl_stats["max"]:.1f}')
        print(f'     Mean: {pdl_stats["mean"]:.1f} | Std: {pdl_stats["std"]:.1f}')
        stats['pdl_min'] = float(pdl_stats['min'])
        stats['pdl_max'] = float(pdl_stats['max'])

    # ── CHECK 7: PDL por donante (detectar si PDL es comparable entre donantes) ──
    if 'donor_id' in df.columns and df['pdl'].notna().sum() > 0:
        print(f'\n  🧑‍🔬 PDL por donante (eje longitudinal):')
        pdl_by_donor = df.groupby('donor_id')['pdl'].agg(['min', 'max', 'count'])
        print(pdl_by_donor.to_string())

    # ── Reporte final ──
    print('\n' + '=' * 60)
    if issues:
        print('\n🚨 ISSUES CRÍTICOS (deben resolverse antes de entrenar):')
        for issue in issues:
            print(f'  {issue}')

    if warnings_list:
        print('\n⚠️  ADVERTENCIAS (revisar antes de entrenar):')
        for w in warnings_list:
            print(f'  {w}')

    if not issues and not warnings_list:
        print('\n✅ Manifest válido — sin issues críticos')

    return stats, issues, warnings_list

stats, issues, warnings_list = validate_manifest(df_manifest)

In [ ]:
# ── Visualización del manifest ─────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle('Sturm Dataset — Manifest QC Report', fontsize=14, fontweight='bold', y=0.98)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    # Plot 1: Disponibilidad de modalidades
    ax1 = fig.add_subplot(gs[0, 0])
    modality_cols = ['has_image', 'has_rna', 'has_methylation', 'has_telomere', 'has_mtdna']
    modality_counts = [df_manifest[c].sum() if c in df_manifest.columns else 0
                       for c in modality_cols]
    labels = [c.replace('has_', '') for c in modality_cols]
    colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
    bars = ax1.barh(labels, modality_counts, color=colors, edgecolor='white')
    ax1.set_xlabel('N muestras')
    ax1.set_title('Disponibilidad por modalidad')
    for bar, count in zip(bars, modality_counts):
        ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 str(count), va='center', fontsize=9)

    # Plot 2: Distribución de n_modalities
    ax2 = fig.add_subplot(gs[0, 1])
    if 'n_modalities' in df_manifest.columns:
        mod_dist = df_manifest['n_modalities'].value_counts().sort_index()
        ax2.bar(mod_dist.index, mod_dist.values, color='#4e79a7', edgecolor='white')
        ax2.set_xlabel('N modalidades disponibles')
        ax2.set_ylabel('N muestras')
        ax2.set_title('Cobertura multimodal por muestra')
        ax2.set_xticks(range(0, 6))

    # Plot 3: PDL por donante
    ax3 = fig.add_subplot(gs[0, 2])
    if 'pdl' in df_manifest.columns and 'donor_id' in df_manifest.columns:
        donors = df_manifest['donor_id'].dropna().unique()
        for i, donor in enumerate(sorted(donors)):
            sub = df_manifest[df_manifest['donor_id'] == donor].sort_values('pdl')
            if 'pdl' in sub.columns and sub['pdl'].notna().sum() > 0:
                ax3.plot(sub['pdl'], [i] * len(sub), 'o-', alpha=0.7, markersize=4,
                         label=str(donor))
        ax3.set_xlabel('PDL')
        ax3.set_yticks(range(len(donors)))
        ax3.set_yticklabels([str(d) for d in sorted(donors)], fontsize=7)
        ax3.set_title('Trayectoria PDL por donante')
        ax3.grid(axis='x', alpha=0.3)

    # Plot 4: Distribución de PDL (histograma)
    ax4 = fig.add_subplot(gs[1, 0])
    if 'pdl' in df_manifest.columns and df_manifest['pdl'].notna().sum() > 0:
        ax4.hist(df_manifest['pdl'].dropna(), bins=30, color='#4e79a7',
                 edgecolor='white', alpha=0.8)
        ax4.set_xlabel('PDL')
        ax4.set_ylabel('N muestras')
        ax4.set_title('Distribución de PDL')
        ax4.axvline(df_manifest['pdl'].mean(), color='red', linestyle='--',
                    label=f'Mean={df_manifest["pdl"].mean():.1f}', alpha=0.7)
        ax4.legend(fontsize=8)

    # Plot 5: Mapa de calor de missingness (modalidades x muestras)
    ax5 = fig.add_subplot(gs[1, 1:])
    modality_cols_present = [c for c in modality_cols if c in df_manifest.columns]
    if modality_cols_present:
        sample_order = df_manifest.sort_values(['donor_id', 'pdl']).index
        missing_matrix = df_manifest.loc[sample_order, modality_cols_present].astype(int).T
        im = ax5.imshow(missing_matrix.values, aspect='auto', cmap='RdYlGn',
                        vmin=0, vmax=1, interpolation='nearest')
        ax5.set_yticks(range(len(modality_cols_present)))
        ax5.set_yticklabels([c.replace('has_', '') for c in modality_cols_present])
        ax5.set_xlabel('Muestras (ordenadas por donante y PDL)')
        ax5.set_title('Missingness multimodal (verde=disponible, rojo=faltante)')
        plt.colorbar(im, ax=ax5, fraction=0.02, pad=0.02)

    plt.savefig(DIRS['reports'] / 'manifest_qc_report.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\n✅ Reporte QC guardado')

except ImportError:
    print('matplotlib no disponible, saltando visualización')

## PASO 7 — Export, Versionado y Reporte Final

In [ ]:
# ── Ordenar columnas del manifest para legibilidad ────────────────────────────
MANIFEST_COLUMN_ORDER = [
    # IDs y metadata de muestra
    'sample_id', 'donor_id', 'passage', 'pdl', 'pdl_norm', 'pdl_bin',
    'phase', 'treatment', 'condition', 'timepoint',

    # Flags de disponibilidad
    'has_image', 'n_images',
    'has_rna',
    'has_methylation',
    'has_telomere',
    'has_mtdna',
    'n_modalities', 'modalities_available',

    # Biomarcadores (anclas de hallmarks)
    'telomere_length', 'mtdna_cn', 'cell_size',

    # Referencias cruzadas
    'geo_rna_sample_id', 'geo_methyl_sample_id',

    # Metadatos del pipeline
    'manifest_version', 'manifest_date',
]

# Incluir solo columnas que existen
final_cols = [c for c in MANIFEST_COLUMN_ORDER if c in df_manifest.columns]
extra_cols = [c for c in df_manifest.columns if c not in MANIFEST_COLUMN_ORDER]
final_cols += extra_cols

df_manifest_final = df_manifest[final_cols]

# ── Guardar manifest versionado ───────────────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
manifest_filename = f'sturm_manifest_v{CONFIG["pipeline_version"]}_{timestamp}.csv'
manifest_path = DIRS['manifests'] / manifest_filename
manifest_latest_path = DIRS['manifests'] / 'manifest_latest.csv'

df_manifest_final.to_csv(manifest_path, index=False)
df_manifest_final.to_csv(manifest_latest_path, index=False)

# Calcular checksum del manifest
manifest_md5 = compute_md5(manifest_path)

print(f'💾 Manifest guardado:')
print(f'   Versionado: {manifest_path}')
print(f'   Latest:     {manifest_latest_path}')
print(f'   MD5: {manifest_md5}')
print(f'   Shape: {df_manifest_final.shape}')

In [ ]:
# ── Reporte final del pipeline ────────────────────────────────────────────────
report = {
    'pipeline_version': CONFIG['pipeline_version'],
    'execution_date': CONFIG['pipeline_date'],
    'manifest_file': manifest_filename,
    'manifest_md5': manifest_md5,
    'manifest_shape': list(df_manifest_final.shape),
    'stats': stats,
    'issues': issues,
    'warnings': warnings_list,
    'sources': {
        'figshare_central': f'https://figshare.com/articles/dataset/{CONFIG["figshare_central_article_id"]}',
        'figshare_images':  f'https://figshare.com/articles/dataset/{CONFIG["figshare_images_article_id"]}',
        'geo_rna':          f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={CONFIG["geo_rna"]}',
        'geo_methylation':  f'https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={CONFIG["geo_methylation"]}',
    }
}

report_path = DIRS['reports'] / f'pipeline_report_{timestamp}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print('📋 REPORTE FINAL DEL PIPELINE')
print('=' * 60)
print(f'  Fecha:         {report["execution_date"]}')
print(f'  Versión:       {report["pipeline_version"]}')
print(f'  Muestras:      {stats.get("n_samples", "N/A")}')
print(f'  Donantes:      {stats.get("n_donors", "N/A")}')
print(f'  Multimodal:    {stats.get("multimodal_samples", "N/A")} muestras con ≥2 modalidades')
print(f'  Issues:        {len(issues)}')
print(f'  Warnings:      {len(warnings_list)}')
print(f'  Manifest MD5:  {manifest_md5[:16]}...')
print(f'\n✅ Pipeline completado. Manifest listo para encoders.')
print(f'\n📁 Archivos generados:')
print(f'   {manifest_path}')
print(f'   {report_path}')
print(f'   {DIRS["reports"]}/manifest_qc_report.png')

## SIGUIENTE PASO: Resolución del join de IDs

Después de correr este notebook, lo más probable es que encuentres que el join entre modalidades no es perfecto (la columna `has_*` muestra muchos False donde debería haber True). Eso es normal y esperado.

**El siguiente notebook** (`02_id_resolution.ipynb`) se encargará de:

1. Inspeccionar los nombres reales de columnas del CSV central de Sturm
2. Ajustar el `COLUMN_MAP` con los nombres reales
3. Inspeccionar el naming convention de las imágenes y crear el parser correcto
4. Hacer el join definitivo entre modalidades

**Para depurar el join, corre esto:**

```python
# Ver todas las columnas del CSV central
print(df_central.columns.tolist())
print(df_central.head(3))

# Ver nombres reales de imágenes
print(df_images['filename'].head(20).tolist())

# Ver características GEO
print(df_rna_meta.head(3))
```